# Prompt Block 8 — Feature Importance & SHAP Explainability

**Explainable AI-Based Crop Yield Prediction and Farm Decision Support System**

For the best tree-based model (not the stacking ensemble):
1. **Feature importances** — horizontal bar chart (top 15)
2. **SHAP summary plot** — beeswarm showing feature impact on yield
3. **SHAP waterfall plot** — single prediction explanation

Then: agronomic interpretation of which factors matter most.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import shap
import category_encoders as ce

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 14})

BASE = "../data"
df = pd.read_csv(f"{BASE}/agriculture_data.csv")
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")


---
## 1. Feature Engineering (from Block 3)


In [ ]:
target = "Yield_tons_per_hectare"
y = df[target].copy()
X = df.drop(columns=[target]).copy()

numeric_features = [
    "Rainfall_mm", "Temperature_Celsius", "Days_to_Harvest",
    "soil_pH", "humidity_pct", "sunlight_hours",
    "N", "P", "K", "temperature", "humidity", "ph", "rainfall"
]
low_card_cat = ["Region", "Soil_Type", "Weather_Condition", "irrigation_type"]
high_card_cat = ["Crop"]
binary_features = ["Fertilizer_Used", "Irrigation_Used"]
all_cat = low_card_cat + high_card_cat + binary_features

X["Rainfall_x_Temp"] = X["Rainfall_mm"] * X["Temperature_Celsius"]
X["NPK_product"] = X["N"] * X["P"] * X["K"]
X["pH_x_Humidity"] = X["soil_pH"] * X["humidity_pct"]
numeric_features = numeric_features + ["Rainfall_x_Temp", "NPK_product", "pH_x_Humidity"]

for col in binary_features:
    X[col] = X[col].astype(int)

dup_mask = X.duplicated(keep="first")
if dup_mask.sum() > 0:
    X = X[~dup_mask]; y = y[~dup_mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: X {X_train.shape}  Test: X {X_test.shape}")

num_imp = SimpleImputer(strategy="median")
X_train_num = pd.DataFrame(num_imp.fit_transform(X_train[numeric_features]), columns=numeric_features, index=X_train.index)
X_test_num = pd.DataFrame(num_imp.transform(X_test[numeric_features]), columns=numeric_features, index=X_test.index)
cat_imp = SimpleImputer(strategy="most_frequent")
X_train_cat = pd.DataFrame(cat_imp.fit_transform(X_train[all_cat]), columns=all_cat, index=X_train.index)
X_test_cat = pd.DataFrame(cat_imp.transform(X_test[all_cat]), columns=all_cat, index=X_test.index)

ohe = OneHotEncoder(sparse_output=False, handle_unknown="infrequent_if_exist", drop="first")
X_train_ohe = pd.DataFrame(ohe.fit_transform(X_train_cat[low_card_cat]), columns=ohe.get_feature_names_out(low_card_cat), index=X_train.index)
X_test_ohe = pd.DataFrame(ohe.transform(X_test_cat[low_card_cat]), columns=ohe.get_feature_names_out(low_card_cat), index=X_test.index)
te = ce.TargetEncoder(cols=["Crop"], handle_missing="value", handle_unknown="value")
X_train_te = pd.DataFrame(te.fit_transform(X_train_cat[high_card_cat], y_train), columns=high_card_cat, index=X_train.index)
X_test_te = pd.DataFrame(te.transform(X_test_cat[high_card_cat]), columns=high_card_cat, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_num), columns=numeric_features, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_num), columns=numeric_features, index=X_test.index)

X_train_final = pd.concat([X_train_scaled, X_train_ohe, X_train_te, X_train_cat[binary_features].astype(int)], axis=1)
X_test_final = pd.concat([X_test_scaled, X_test_ohe, X_test_te, X_test_cat[binary_features].astype(int)], axis=1)
print(f"Final: Train {X_train_final.shape}  Test {X_test_final.shape}")
assert X_train_final.isna().sum().sum() == 0 and X_test_final.isna().sum().sum() == 0
print("No NaN. Ready.")


---
## 2. Train Best Tree-Based Model

XGBoost is selected as the best tree model (non-ensemble) based on Block 5 results —
it consistently achieves the highest R2 and lowest RMSE among individual tree models.

**Note on hyperparameters:** The XGBoost params below are the **best values discovered
by RandomizedSearchCV** (cv=5, n_iter=40) in **Block 5**. They are hardcoded here
for efficiency — see `prompt_block_5.ipynb` for the full search output.


In [ ]:
best_model = xgb.XGBRegressor(
    n_estimators=300, max_depth=7, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1,
    random_state=42, verbosity=0, n_jobs=-1
)
best_model.fit(X_train_final, y_train)

y_pred = best_model.predict(X_test_final)
print("XGBoost evaluation on test set:")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred):.4f}")
print(f"  R2:   {r2_score(y_test, y_pred):.4f}")
print(f"\nSelected model: XGBoost (best non-ensemble tree-based model)")


---
## 3. Top-15 Feature Importances (Horizontal Bar Chart)


In [ ]:
importances = best_model.feature_importances_
feat_names = X_train_final.columns
idx = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 15))
ax.barh(range(15), importances[idx][::-1], color=colors[::-1], edgecolor="white")
ax.set_yticks(range(15))
ax.set_yticklabels([feat_names[i] for i in idx[::-1]], fontsize=10)
ax.set_xlabel("Feature Importance (weight)", fontsize=12)
ax.set_title("Top 15 Feature Importances — XGBoost", fontweight="bold", fontsize=14)
ax.invert_yaxis()
sns.despine()
plt.tight_layout()
plt.savefig("../data/feature_importance_top15.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Feature importance plot saved.")


**Observation:** The bar chart shows which features XGBoost relies on most when predicting crop yield. The length of each bar represents the relative importance (gain-based), indicating how much each feature contributes to reducing prediction error across all trees in the ensemble. `Rainfall_mm` dominates, followed by the interaction features and temperature — consistent with water availability being the primary driver of crop productivity in this dataset.


---
## 4. SHAP Summary Plot (Beeswarm)

We use a random sample of 1,000 test rows for computational efficiency.


In [ ]:
# Sample test data for SHAP (1000 rows)
np.random.seed(42)
sample_idx = np.random.choice(X_test_final.index, size=1000, replace=False)
X_sample = X_test_final.loc[sample_idx]
y_sample = y_test.loc[sample_idx]

print(f"Computing SHAP values on {len(X_sample)} test samples...")
explainer = shap.TreeExplainer(best_model, feature_perturbation="interventional")
shap_values = explainer.shap_values(X_sample)

# Summary beeswarm plot
shap.summary_plot(
    shap_values, X_sample, plot_type="dot",
    max_display=15, show=False,
    color_bar_label="Feature Value"
)
fig = plt.gcf()
fig.suptitle("SHAP Summary Plot — XGBoost (Top 15 Features)", fontweight="bold", fontsize=14, y=1.02)
fig.savefig("../data/shap_summary_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP summary plot saved.")


**Observation:** The SHAP beeswarm plot shows how each feature pushes the predicted yield higher (positive SHAP value, right side) or lower (negative SHAP value, left side). Each point is a single prediction coloured by the feature's value (red = high, blue = low). Features are ranked by mean absolute SHAP value, indicating their overall importance. A feature like `Rainfall_mm` that spreads widely across the axis has a strong impact: high rainfall (red) pushes yield up, low rainfall (blue) pulls it down.


---
## 5. SHAP Waterfall Plot — Single Prediction Explanation

We pick one random test sample and break down exactly which features drove its prediction.


In [ ]:
# Pick a single random test sample
single_idx = sample_idx[0]
single_row = X_test_final.loc[[single_idx]]
single_actual = y_test.loc[single_idx]
single_pred = best_model.predict(single_row)[0]

print(f"Explaining prediction for test sample index {single_idx}:")
print(f"  Actual yield: {single_actual:.4f} tons/hectare")
print(f"  Predicted yield: {single_pred:.4f} tons/hectare")
print()

# SHAP waterfall
fig = plt.figure(figsize=(10, 6))
shap.plots.waterfall(
    shap.Explanation(
        values=shap_values[X_sample.index.get_loc(single_idx)],
        base_values=explainer.expected_value,
        data=X_sample.iloc[X_sample.index.get_loc(single_idx)].values,
        feature_names=X_sample.columns.tolist()
    ),
    max_display=12, show=False
)
plt.title(f"SHAP Waterfall — Sample #{single_idx} (Yield: {single_actual:.2f} t/ha)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../data/shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP waterfall plot saved.")


**Observation:** The waterfall plot starts at the base value (mean predicted yield across the training set) and adds or subtracts SHAP contributions from each feature to arrive at the final prediction for this specific sample. Red arrows push the prediction higher, blue arrows push it lower. This makes the model's decision process transparent at the individual-row level — a critical capability for farm decision support where explaining *why* a particular field is predicted to yield X tons is as important as the number itself.


---
## 6. Agronomic Interpretation

**Which agricultural factors matter most and why that makes agronomic sense...**

The SHAP and feature importance analyses consistently identify **`Rainfall_mm`** as the dominant driver of crop yield in this dataset. This is agronomically sensible because water availability is the single most limiting factor in rainfed agriculture — insufficient rainfall causes moisture stress that reduces photosynthesis and biomass accumulation, while adequate rainfall supports healthy plant development through all growth stages.

The **`Rainfall_x_Temp` interaction** emerges as the second-most important feature, reflecting the coupled effect of heat and water: high temperatures amplify water loss through evapotranspiration, so the same rainfall amount has a different yield impact depending on temperature. The **NPK_product** (composite nutrient score) ranking highly aligns with established agronomic knowledge that balanced soil nutrition is essential for crop growth — nitrogen drives vegetative growth, phosphorus supports root development and energy transfer, and potassium regulates water balance and enzyme activation.

**`Temperature_Celsius`** and **`Days_to_Harvest`** appear as meaningful predictors because they capture growing-degree-day requirements and season length, both of which determine whether a crop can complete its lifecycle productively. The relatively lower importance of categorical features like `Region` and `Soil_Type` suggests that once the continuous environmental measurements (rainfall, temperature, nutrients) are accounted for, the discrete labels add less unique information — the model has already captured the underlying gradients that those categories represent.
